In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 8.3 Enhancing Predictive Models with Survey Data I
- Reuse tuned XGBoost hyperparameters, refit on survey-enhanced features
- Standard classification metrics
- Feature importance by group: Academic vs Demographics vs Survey/Text

## Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
import matplotlib.pyplot as plt

train_df = pd.read_csv('../data/ML_SURVEY_MASTER_TRAIN.csv')
test_df = pd.read_csv('../data/ML_SURVEY_MASTER_TEST.csv')

train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

X_train = train_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])
y_train = train_df['DEPARTED']
X_test = test_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])
y_test = test_df['DEPARTED']

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## Load Tuned Hyperparameters, Refit on Survey Features
We reuse the tuned XGBoost from Module 3 as a hyperparameter template, then refit it on the richer survey-enhanced feature set (this is why `.fit()` is called again right after loading).

In [ ]:
# Loads the tuned model from Module 3 (produced by 3.3_lesson.ipynb)
xgb_model = pickle.load(open('../models/xgb_tuned_f1.pkl', 'rb'))
xgb_model.fit(X_train, y_train)

## Evaluate

In [ ]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred, zero_division=0):.3f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, zero_division=0))

## Feature Importance by Group
Did survey/text data add signal beyond academics and demographics?

In [ ]:
fi = (pd.DataFrame({"feature": X_train.columns, "importance": xgb_model.feature_importances_})
        .sort_values("importance", ascending=False).reset_index(drop=True))

academic_prefixes = ["HS_GPA", "GPA_", "DFW_RATE_", "UNITS_ATTEMPTED_"]
demo_prefixes = ["GENDER_", "RACE_ETHNICITY_", "FIRST_GEN_STATUS_"]
text_prefixes = ["TEXT_PC"]

def assign_group(col):
    if any(col.startswith(p) for p in academic_prefixes):
        return "Academic performance"
    if any(col.startswith(p) for p in demo_prefixes):
        return "Demographics"
    if any(col.startswith(p) for p in text_prefixes):
        return "Survey/Text components"
    return "Other (check)"

fi["group"] = fi["feature"].apply(assign_group)
group_importance = fi.groupby("group")["importance"].sum().sort_values(ascending=False).reset_index()
print(group_importance)

## Summary
- Reusing tuned hyperparameters and refitting on richer features is a legitimate pattern — the hyperparameters transfer, the model itself doesn't need to be retuned from scratch.
- Feature-group importance answers the real institutional question: did adding survey/text data actually help, and by how much?

**Next:** 8.4 compares this survey-enhanced model against the original 4.1 model family.